# MLflow & database

Тестирование подключения к MLFLow и БД из Jupyter Notebook

## Configuration

In [1]:
# Хак, чтобы добавить корень проекта в path для импорта модулей
import sys
from pathlib import Path

def find_repo_root() -> Path:
    path = Path.cwd().resolve()
    for candidate in [path, *path.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not find repo root (pyproject.toml). Open this notebook from the repo.")

sys.path.append(str(find_repo_root()))
sys.path

['/home/fiberfox/.local/share/uv/python/cpython-3.12.7-linux-x86_64-gnu/lib/python312.zip',
 '/home/fiberfox/.local/share/uv/python/cpython-3.12.7-linux-x86_64-gnu/lib/python3.12',
 '/home/fiberfox/.local/share/uv/python/cpython-3.12.7-linux-x86_64-gnu/lib/python3.12/lib-dynload',
 '',
 '/home/fiberfox/Projects/HSEAIMag2025/stocks-advisor/.venv/lib/python3.12/site-packages',
 '/home/fiberfox/Projects/HSEAIMag2025/stocks-advisor']

In [2]:
# Код для загрузки конфига и подключения к MLFlow
# Для работы с продом в корне проекта должен быть .env файл с всеми нужными переменными окружения
# Для локальной разработки .env не нужен
from jupyter_utils import setup_jupyter_notebook

# Подключение к проду
setup_jupyter_notebook(environment='prod', experiment='test')

# Подключение к локальным БД и MLFlow
# setup_jupyter_notebook(environment='local', experiment='test')

Environment: prod
APP_CONFIG: config.toml
Tracking URI: http://localhost:5050
S3 endpoint: http://localhost:9050
Experiment: test
Database: localhost:15432/stocks_advisor_db


## MLFlow

Пример логгирования эксперимента в MLFlow

In [3]:
from datetime import UTC, datetime
import mlflow

run_name = f"smoke-test-{datetime.now(UTC).strftime('%Y%m%d-%H%M%S')}"

with mlflow.start_run(run_name=run_name) as run:
    mlflow.log_param("source", "mlflow_smoke_test")
    mlflow.log_metric("ping", 1.0)
    mlflow.log_dict({"status": "ok"}, "smoke.json")

print(f"Run ID: {run.info.run_id}")
print(f"Run name: {run_name}")
print("Smoke test passed.")

🏃 View run smoke-test-20260531-135823 at: http://localhost:5050/#/experiments/2/runs/dba9279e49de4912940c5149e927b0ec
🧪 View experiment at: http://localhost:5050/#/experiments/2
Run ID: dba9279e49de4912940c5149e927b0ec
Run name: smoke-test-20260531-135823
Smoke test passed.


## Database queries

Примеры загрузки данных из PostgreSQL:

- Стоимость акций
- Данные по новостям: тикеры, сектор, sentiment

In [4]:
# Загрузка свечей MOEX

from app.core.database import AssetCandleRepository, get_db_session

async with get_db_session() as session:
    repo = AssetCandleRepository(session)
    candles_by_ticker = await repo.get_dataframe_by_ticker(date_start=datetime(2026, 1, 1))

print(f'{len(candles_by_ticker)} tickers')
for ticker, df in candles_by_ticker.items():
    print(f'  {ticker}: {len(df)} rows')

11 tickers
  BRENT: 1572 rows
  CNYRUBF: 1545 rows
  EURRUBF: 1470 rows
  GAZP: 2056 rows
  GLDRUB_TOM: 970 rows
  IMOEX: 989 rows
  LKOH: 2053 rows
  ROSN: 2053 rows
  SBER: 2056 rows
  T: 1958 rows
  USDRUBF: 1535 rows


In [5]:
# Генерация фич

from app.core.processors.feature_generator import FeatureGenerator


features_by_ticker = {}
fg = FeatureGenerator()

for ticker, df in candles_by_ticker.items():
    features_by_ticker[ticker] = fg.process(
        df=df,
        include_original=False,
        add_targets=False,
        clean=True,
    )
features_by_ticker['SBER'].head()

,begin,MA_20,MA_50,MA_90,MA_200,EMA_20,EMA_50,EMA_90,EMA_200,WMA_20,...,DISTANCE_MA_90,DISTANCE_MA_200,DISTANCE_EMA_20,DISTANCE_EMA_50,DISTANCE_EMA_90,DISTANCE_EMA_200,DISTANCE_WMA_20,DISTANCE_WMA_50,DISTANCE_WMA_90,DISTANCE_WMA_200
200,2026-02-19 10:00:00,310.7950,306.3230,305.281333,303.69165,310.732996,307.868505,306.374427,305.036261,311.693190,...,0.028756,0.034141,0.010707,0.020111,0.025086,0.029583,0.007593,0.017870,0.025063,0.029248
201,2026-02-19 12:00:00,311.0680,306.5496,305.363111,303.76745,310.969853,308.078433,306.526588,305.130146,311.924143,...,0.025730,0.031118,0.007236,0.016689,0.021836,0.026513,0.004154,0.014259,0.021740,0.026176
202,2026-02-19 14:00:00,311.3735,306.7708,305.449889,303.84070,311.155582,308.268354,306.668663,305.219376,312.100524,...,0.024456,0.029882,0.005671,0.015090,0.020385,0.025230,0.002626,0.012468,0.020208,0.024887
203,2026-02-19 16:00:00,311.6090,306.9940,305.533111,303.91000,311.322669,308.450432,306.807325,305.307336,312.246857,...,0.024144,0.029614,0.005099,0.014458,0.019891,0.024902,0.002124,0.011648,0.019631,0.024552
204,2026-02-19 18:00:00,311.8550,307.2182,305.621000,303.97980,311.478605,308.627326,306.943984,305.394733,312.375524,...,0.024013,0.029542,0.004756,0.014039,0.019600,0.024772,0.001871,0.011045,0.019251,0.024414


In [6]:
# Загрузка данных по новостям

from app.core.database import NewsArticleRepository, get_db_session

LIMIT = 100

async with get_db_session() as session:
    repo = NewsArticleRepository(session)
    enrichments_df = await repo.get_all_enrichments_as_dataframe(limit=LIMIT, ticker='SBER', sector='MOEXFN', date_start=datetime(2026, 1, 1))

print(f'Enrichments: {len(enrichments_df)} rows')
enrichments_df.head()

Enrichments: 100 rows


,id,news_article_id,published_at,topic,tickers,sector,sentiment,sentiment_score
0,40805,198675,2026-01-06 10:29:00,business,"[SBER, ROSN, GAZP, LKOH, TATN]",MOEXFN,positive,0.890096
1,40816,198667,2026-01-06 16:59:00,business,"[SBER, YDEX, ROSN, GAZP, LKOH, TATN, VTBR]",MOEXFN,positive,0.851830
2,40825,198661,2026-01-06 19:00:00,business,"[SBER, YDEX, ROSN, GAZP, LKOH, TATN, VTBR]",MOEXFN,negative,0.865966
3,40924,198684,2026-01-08 19:25:00,business,"[SBER, YDEX, ROSN, GAZP, LKOH, TATN, VTBR]",MOEXFN,negative,0.796051
4,40960,198725,2026-01-09 10:07:00,business,"[SBER, YDEX, ROSN, GAZP, LKOH, TATN, VTBR]",MOEXFN,positive,0.978370
